# 06 — Mitigation: GRU + Reweighting + Adversarial Debiasing (Combined)

Stacks the two mitigations from notebooks 04–05 into a single training run:

| Layer | Method | Effect |
|-------|--------|--------|
| Pre-processing | Importance-weight reweighting | Corrects *gradient signal* — aligns per-group toxic rates toward corpus mean |
| In-processing | Adversarial debiasing (GRL) | Corrects *embedding space* — penalises encoder for retaining identity-predictive features |

The toxicity loss is importance-weighted (notebook 04); the adversary loss is added on top (notebook 05). The hypothesis is that each method addresses a distinct failure mode, so stacking them should improve on both individually.

| Step | Description |
|------|-------------|
| 1 | Load data |
| 2 | Text preprocessing |
| 3 | Importance weights |
| 4 | Adversarial model (FairGRU) |
| 5 | Combined training |
| 6 | Inference & overall performance |
| 7 | Fairness evaluation |
| 8 | Counterfactual gap |
| 9 | Conclusion |

> **Authorship note.** Code generation and prose editing in this notebook were assisted by Claude Sonnet. Method selection, limitation analysis, and the substance of result interpretations were completed by the human author.

## 0. Setup

In [1]:
import warnings; warnings.filterwarnings("ignore")
import json, math, numpy as np, pandas as pd, torch, torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.autograd import Function
from sklearn.metrics import classification_report, roc_auc_score
from transformers import BertTokenizer
from fairness_jigsaw.metrics import DEFAULT_IDENTITY_COLUMNS, ModelBiasEvaluator

DEVICE = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)
print(f"Device: {DEVICE}")
SEED = 42; torch.manual_seed(SEED); np.random.seed(SEED)
TOXICITY_THRESHOLD = 0.5; IDENTITY_THRESHOLD = 0.5
N_TRAIN_SAMPLE = None
MAX_LEN = 128
EMBED_DIM = 64; HIDDEN_DIM = 128
BATCH_SIZE = 256; EPOCHS = 3; LR = 1e-3
LAMBDA_ADV = 0.5
ADV_ALPHA  = 2.0

Device: mps


## 1. Data

Identical splits to all previous notebooks (`data/split_ids.json`, seed 1337).

In [2]:
with open("../data/split_ids.json") as f:
    split_ids = json.load(f)
USE_COLS = ["id", "target", "comment_text"] + list(DEFAULT_IDENTITY_COLUMNS)
df_all = pd.read_csv("../data/train.csv", usecols=USE_COLS)
df_all["toxic"] = (df_all["target"] >= TOXICITY_THRESHOLD).astype(int)
train_df = df_all[df_all["id"].isin(split_ids["train"])].reset_index(drop=True)
val_df   = df_all[df_all["id"].isin(split_ids["val"])].reset_index(drop=True)
test_df  = df_all[df_all["id"].isin(split_ids["test"])].reset_index(drop=True)
if N_TRAIN_SAMPLE is not None:
    train_df = train_df.sample(n=N_TRAIN_SAMPLE, random_state=SEED).reset_index(drop=True)
for name, df in [("Train", train_df), ("Val  ", val_df), ("Test ", test_df)]:
    anno = df[list(DEFAULT_IDENTITY_COLUMNS)].notna().any(axis=1).sum()
    print(f"{name}: {len(df):>10,}  | toxic: {df['toxic'].mean():.2%}  | annotated: {anno:>7,} ({anno/len(df):.1%})")

Train:  1,443,897  | toxic: 8.00%  | annotated: 324,097 (22.4%)
Val  :    180,486  | toxic: 8.00%  | annotated:  40,486 (22.4%)
Test :    180,491  | toxic: 8.00%  | annotated:  40,547 (22.5%)


## 2. Text Preprocessing

BERT WordPiece tokenizer (`bert-base-uncased`), truncated/padded to `MAX_LEN`.

In [3]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
print(f"Vocabulary size : {tokenizer.vocab_size:,}")

def encode(text) -> list[int]:
    return tokenizer.encode(
        text if isinstance(text, str) else "",
        max_length=MAX_LEN, truncation=True, padding="max_length",
    )

class CommentDataset(Dataset):
    def __init__(self, df: pd.DataFrame) -> None:
        self.x = torch.tensor([encode(t) for t in df["comment_text"]], dtype=torch.long)
        self.y = torch.tensor(df["toxic"].values, dtype=torch.float32)
    def __len__(self): return len(self.x)
    def __getitem__(self, idx): return self.x[idx], self.y[idx]

test_loader = DataLoader(CommentDataset(test_df), batch_size=BATCH_SIZE)
print(f"Test batches: {len(test_loader)}")

Vocabulary size : 30,522
Test batches: 706


## 3. Importance Weights

Same toxicity-rate reweighting as notebook 04.

- **w_toxic(c)** = `tox_mean / tox_rate(c)` — downscales toxic loss for over-toxic groups.
- **w_nontoxic(c)** = `(1 − tox_mean) / (1 − tox_rate(c))` — upscales non-toxic loss for over-toxic groups.

Per-comment weight = max over belonging identities; unannotated comments keep weight 1.0.

In [4]:
for col in DEFAULT_IDENTITY_COLUMNS:
    train_df[f"{col}_bin"] = (train_df[col].fillna(0) >= IDENTITY_THRESHOLD).astype(int)

def compute_importance_weights(
    df: pd.DataFrame,
    identity_cols: list,
    label_col: str = "toxic",
    clip: tuple = (0.1, 10.0),
) -> tuple:
    tox_mean = float(df[label_col].mean())
    labels   = df[label_col].to_numpy()
    records, wtox_list, wntox_list = [], [], []
    for col in identity_cols:
        mask = df[f"{col}_bin"].astype(bool)
        n    = int(mask.sum())
        if n == 0:
            tox_rate, wtox, wntox = float("nan"), 1.0, 1.0
        else:
            tox_rate = float(df.loc[mask, label_col].mean())
            wtox  = float(np.clip(tox_mean / tox_rate           if tox_rate > 0   else 1.0, *clip))
            wntox = float(np.clip((1-tox_mean) / (1-tox_rate) if tox_rate < 1.0 else 1.0, *clip))
        records.append({"identity": col, "tox_rate": tox_rate,
                        "tox_target": tox_mean, "w_toxic": wtox, "w_nontoxic": wntox})
        wtox_list.append(wtox); wntox_list.append(wntox)
    iw_df = pd.DataFrame(records).sort_values("tox_rate", ascending=False).reset_index(drop=True)
    bin_mat   = df[[f"{col}_bin" for col in identity_cols]].to_numpy(dtype=np.float32)
    has_id    = bin_mat.max(axis=1) > 0
    wtox_arr  = np.array(wtox_list,  dtype=np.float32)
    wntox_arr = np.array(wntox_list, dtype=np.float32)
    w_arr       = np.where(labels[:, None] == 1, wtox_arr, wntox_arr)
    per_comment = np.where(has_id, (bin_mat * w_arr).max(axis=1), 1.0).astype(np.float32)
    return iw_df, per_comment

iw_df, train_weights = compute_importance_weights(
    train_df, list(DEFAULT_IDENTITY_COLUMNS), label_col="toxic"
)
print("Weight distribution:")
print(pd.Series(train_weights).describe(percentiles=[0.5, 0.9, 0.99]).to_string())

Weight distribution:
count    1.443897e+06
mean     1.002487e+00
std      8.197168e-02
min      2.535932e-01
50%      1.000000e+00
90%      1.000000e+00
99%      1.276429e+00
max      1.459422e+00


## 4. Adversarial Model

Same `FairGRU` + Gradient Reversal Layer as notebook 05.

```
Embedding → GRU → rep ─┬→ Linear → Sigmoid        (toxicity classifier)
                       └→ GRL(α) → MLP → Sigmoid  (identity adversary)
```

The GRL negates adversary gradients before they reach the encoder, penalising identity-predictive representations.

In [5]:
class GRL(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class FairGRU(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int, n_identities: int) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.classifier = nn.Linear(hidden_dim, 1)
        self.adversary = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, n_identities),
        )

    def forward(self, x: torch.Tensor, alpha: float = 0.0):
        _, h = self.gru(self.embedding(x))
        rep  = h[-1]
        tox  = torch.sigmoid(self.classifier(rep)).squeeze(-1)
        adv  = torch.sigmoid(self.adversary(GRL.apply(rep, alpha)))
        return tox, adv

model = FairGRU(tokenizer.vocab_size, EMBED_DIM, HIDDEN_DIM, len(DEFAULT_IDENTITY_COLUMNS)).to(DEVICE)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

FairGRU(
  (embedding): Embedding(30522, 64, padding_idx=0)
  (gru): GRU(64, 128, batch_first=True)
  (classifier): Linear(in_features=128, out_features=1, bias=True)
  (adversary): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=24, bias=True)
  )
)

Total parameters: 2,037,849


## 5. Combined Training

`WeightedAdvCommentDataset` carries three targets per sample:
- `y_tox` — binary toxicity label
- `y_id`  — binarised identity columns (adversary target)
- `w`     — per-sample importance weight

The combined loss is:

$$\mathcal{L} = \underbrace{\text{BCE}(\hat{y}_{\text{tox}},\, y_{\text{tox}},\, w)}_{\text{weighted classification}} + \underbrace{\text{BCE}(\hat{y}_{\text{id}},\, y_{\text{id}})}_{\text{adversary (annotated rows only)}}$$

The GRL alpha follows the same sigmoid ramp as notebook 05.

In [6]:
class WeightedAdvCommentDataset(Dataset):
    """Combines per-sample importance weights (pre-processing) with identity labels (adversary)."""
    def __init__(self, df: pd.DataFrame, weights: np.ndarray) -> None:
        self.x     = torch.tensor([encode(t) for t in df["comment_text"]], dtype=torch.long)
        self.y_tox = torch.tensor(df["toxic"].values, dtype=torch.float32)
        bin_cols   = [f"{c}_bin" for c in DEFAULT_IDENTITY_COLUMNS]
        self.y_id  = torch.tensor(df[bin_cols].fillna(0).values, dtype=torch.float32)
        self.w     = torch.tensor(weights, dtype=torch.float32)

    def __len__(self): return len(self.x)
    def __getitem__(self, idx): return self.x[idx], self.y_tox[idx], self.y_id[idx], self.w[idx]

train_loader = DataLoader(
    WeightedAdvCommentDataset(train_df, train_weights),
    batch_size=BATCH_SIZE, shuffle=True,
)
print(f"Train batches: {len(train_loader)}  |  Test batches: {len(test_loader)}")

Train batches: 5641  |  Test batches: 706


In [7]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

for epoch in range(1, EPOCHS + 1):
    progress = (epoch - 1) / max(EPOCHS - 1, 1)
    alpha    = LAMBDA_ADV * (2 / (1 + math.exp(-ADV_ALPHA * progress)) - 1)

    model.train()
    total_tox, total_adv, n_batches = 0.0, 0.0, 0
    for x, y_tox, y_id, w in train_loader:
        x, y_tox, y_id, w = x.to(DEVICE), y_tox.to(DEVICE), y_id.to(DEVICE), w.to(DEVICE)
        optimizer.zero_grad()
        tox_pred, adv_pred = model(x, alpha=alpha)

        # Importance-weighted toxicity loss (pre-processing)
        L_tox = F.binary_cross_entropy(tox_pred, y_tox, weight=w)

        # Adversary loss on annotated rows only (in-processing)
        has_id = y_id.max(dim=1).values > 0
        if has_id.any():
            L_adv = F.binary_cross_entropy(adv_pred[has_id], y_id[has_id])
            loss  = L_tox + L_adv
        else:
            L_adv = tox_pred.new_tensor(0.0)
            loss  = L_tox

        loss.backward()
        optimizer.step()
        total_tox += L_tox.item()
        total_adv += L_adv.item()
        n_batches += 1
    print(f"Epoch {epoch}/{EPOCHS}  α={alpha:.3f}  tox_loss={total_tox/n_batches:.4f}  adv_loss={total_adv/n_batches:.4f}")

Epoch 1/3  α=0.000  tox_loss=0.1549  adv_loss=0.1579
Epoch 2/3  α=0.231  tox_loss=0.1195  adv_loss=0.1584
Epoch 3/3  α=0.381  tox_loss=0.1122  adv_loss=0.1585


In [8]:
import os

save_dir = "../models/gru/reweighting_plus_fair_representation_learning"
os.makedirs(save_dir, exist_ok=True)

torch.save(model.state_dict(), f"{save_dir}/model.pt")
torch.save({
    "embed_dim":    EMBED_DIM,
    "hidden_dim":   HIDDEN_DIM,
    "vocab_size":   tokenizer.vocab_size,
    "n_identities": len(DEFAULT_IDENTITY_COLUMNS),
    "lambda_adv":   LAMBDA_ADV,
    "adv_alpha":    ADV_ALPHA,
    "max_len":      MAX_LEN,
}, f"{save_dir}/config.pt")

print(f"Saved to {save_dir}/")

Saved to ../models/gru/reweighting_plus_fair_representation_learning/


## 6. Inference & Overall Performance

In [9]:
model.eval()
raw_scores: list[float] = []
with torch.no_grad():
    for x, _ in test_loader:
        raw_scores.extend(model(x.to(DEVICE), alpha=0.0)[0].cpu().tolist())

test_df = test_df.copy()
test_df["score"] = raw_scores

overall_auc = roc_auc_score(test_df["toxic"], test_df["score"])
pred_labels = (test_df["score"] >= TOXICITY_THRESHOLD).astype(int)
print(f"Overall AUC          : {overall_auc:.4f}")
print(f"Predicted toxic rate : {pred_labels.mean():.2%}")
print(f"True toxic rate      : {test_df['toxic'].mean():.2%}\n")
print(classification_report(test_df["toxic"], pred_labels, target_names=["non-toxic", "toxic"]))

Overall AUC          : 0.9497
Predicted toxic rate : 5.70%
True toxic rate      : 8.00%

              precision    recall  f1-score   support

   non-toxic       0.96      0.99      0.97    166056
       toxic       0.76      0.55      0.64     14435

    accuracy                           0.95    180491
   macro avg       0.86      0.77      0.80    180491
weighted avg       0.95      0.95      0.95    180491



## 7. Fairness Evaluation

| Metric | What it captures |
|--------|------------------|
| **Subgroup / BPSN / BNSP / Pinned AUC** | Discrimination quality per identity |
| **Subgroup FPR (+ gap)** | Over-flagging of non-toxic comments |
| **ECE** | Calibration quality |

Tables sorted worst-first. Compare to `01_baseline_gru.ipynb`, `04_mitigation_gru_preprocessing.ipynb`, and `05_mitigation_gru_inprocessing.ipynb`.

In [10]:
evaluator = ModelBiasEvaluator(
    identity_cols=DEFAULT_IDENTITY_COLUMNS,
    toxicity_threshold=TOXICITY_THRESHOLD,
    identity_threshold=IDENTITY_THRESHOLD,
    min_subgroup_size=20,
)
results = evaluator.evaluate(test_df, score_col="score", label_col="toxic")

### AUC Metrics

Low BPSN AUC → model over-predicts toxicity for non-toxic comments in the group; low BNSP AUC → under-predicts.

In [11]:
results["auc"].round(4)

,identity,n,subgroup_auc,bpsn_auc,bnsp_auc,pinned_auc
0,hindu,55,0.7154,0.9041,0.8753,0.8039
1,other_religion,34,0.7448,0.9059,0.8958,0.8274
2,buddhist,49,0.7791,0.9084,0.9025,0.8498
3,black,1496,0.7883,0.8876,0.9131,0.8521
4,heterosexual,116,0.7969,0.9190,0.8857,0.8575
5,homosexual_gay_or_lesbian,1141,0.8046,0.8959,0.9141,0.8631
6,muslim,2151,0.8125,0.8966,0.9155,0.8675
7,white,2551,0.8212,0.8942,0.9236,0.8731
8,other_race_or_ethnicity,42,0.8245,0.9116,0.9105,0.8762
9,transgender,255,0.8279,0.9023,0.9154,0.8765


### Subgroup FPR

Positive `fpr_gap` = model over-triggers on non-toxic content from that identity.

In [12]:
results["fpr"].round(4)

,identity,n_negatives,fpr,bg_fpr,fpr_gap
0,transgender,206,0.0437,0.0146,0.0291
1,hindu,47,0.0426,0.0146,0.0279
2,atheist,108,0.0370,0.0146,0.0224
3,psychiatric_or_mental_illness,365,0.0356,0.0146,0.0210
4,other_race_or_ethnicity,35,0.0286,0.0146,0.0140
5,black,1039,0.0269,0.0145,0.0124
6,homosexual_gay_or_lesbian,819,0.0256,0.0146,0.0111
7,male,3801,0.0242,0.0144,0.0098
8,white,1839,0.0228,0.0145,0.0083
9,female,4686,0.0201,0.0145,0.0056


### Expected Calibration Error

First row is overall ECE. Subgroup ECE well above overall indicates miscalibration.

In [13]:
results["ece"].round(4)

,identity,n,ece
0,black,1496,0.1500
1,homosexual_gay_or_lesbian,1141,0.1413
2,heterosexual,116,0.1341
3,white,2551,0.1265
4,other_race_or_ethnicity,42,0.1140
5,other_religion,34,0.1060
6,transgender,255,0.1031
7,latino,209,0.1026
8,hindu,55,0.1024
9,muslim,2151,0.0941


## 8. Counterfactual Gap

Neutral substitution (identity → `"person"`) and pairwise swaps measure the absolute effect of identity tokens on model scores. Compare to previous notebooks to check whether the combined method reduces embedding-level associations.

> **Note.** Only single-word identity terms that appear literally in text are matched. Compound column names (e.g. `homosexual_gay_or_lesbian`) are silently skipped.

In [14]:
def predict_fn(texts: list[str]) -> np.ndarray:
    ids = torch.tensor([encode(t) for t in texts], dtype=torch.long)
    model.eval()
    chunks: list[np.ndarray] = []
    with torch.no_grad():
        for i in range(0, len(ids), BATCH_SIZE):
            chunks.append(model(ids[i:i+BATCH_SIZE].to(DEVICE), alpha=0.0)[0].cpu().numpy())
    return np.concatenate(chunks)

cf_results = evaluator.compute_counterfactual_gap(
    test_df,
    text_col="comment_text",
    score_col="score",
    predict_fn=predict_fn,
    swap_pairs=[("black", "white"), ("christian", "muslim"), ("male", "female")],
)
cf_results.round(4)

,type,term_a,term_b,n_pairs,mean_gap,max_gap
0,neutral,christian,person,991,0.0339,0.3945
1,neutral,black,person,1917,0.0287,0.7514
2,neutral,transgender,person,160,0.0277,0.3253
3,neutral,white,person,3875,0.0208,0.5203
4,neutral,muslim,person,1046,0.0203,0.4791
5,neutral,asian,person,212,0.0139,0.2356
6,neutral,hindu,person,33,0.0122,0.2198
7,neutral,female,person,699,0.0082,0.3018
8,neutral,male,person,827,0.0079,0.3432
9,neutral,jewish,person,339,0.0075,0.0820


## 9. Conclusion

**Four-model comparison — averages across 18 subgroups.**

| Metric | Baseline | Reweighting | Adversarial | Combined |
|--------|:--------:|:-----------:|:-----------:|:--------:|
| Overall AUC | **0.9540** | 0.9524 | 0.9527 | 0.9497 |
| Predicted toxic rate | 5.30% | 4.80% | 5.96% | 5.70% |
| Toxic recall | 53% | 49% | **57%** | 55% |
| *Avg subgroup AUC* | **0.847** | 0.837 | 0.851 | 0.833 |
| *Avg BPSN AUC* | 0.876 | 0.904 | 0.880 | **0.905** |
| *Avg BNSP AUC* | **0.946** | 0.926 | **0.946** | 0.920 |
| *Avg pinned AUC* | 0.881 | 0.881 | **0.885** | 0.879 |
| *Avg FPR* | 0.033 | **0.013** | 0.039 | 0.022 |
| *Avg FPR gap* | 0.022 | **0.003** | 0.024 | 0.007 |
| Overall ECE | 0.006 | 0.012 | **0.003** | 0.007 |
| *Avg subgroup ECE* | **0.052** | 0.089 | 0.058 | 0.083 |
| *Avg neutral CF gap* | 0.028 | **0.014** | 0.032 | **0.014** |

**Selected identity metrics (Baseline / Reweighting / Adversarial / Combined; bold = best).**

| Identity | BPSN AUC | FPR gap | ECE |
|----------|:--------:|:-------:|:---:|
| `black` | 0.806 / 0.874 / 0.823 / **0.888** | +0.059 / **+0.009** / +0.066 / +0.012 | 0.059 / 0.161 / **0.057** / 0.150 |
| `homosexual_gay_or_lesbian` | 0.806 / 0.863 / 0.849 / **0.896** | +0.051 / **+0.008** / +0.040 / +0.011 | **0.038** / 0.133 / 0.067 / 0.141 |
| `transgender` | 0.845 / 0.878 / 0.865 / **0.902** | +0.051 / **+0.020** / +0.053 / +0.029 | **0.062** / 0.091 / 0.069 / 0.103 |
| `muslim` | 0.840 / 0.885 / 0.857 / **0.897** | +0.022 / **−0.002** / +0.033 / +0.002 | **0.034** / 0.105 / 0.036 / 0.094 |
| `white` | 0.823 / 0.886 / 0.843 / **0.894** | +0.034 / **+0.003** / +0.042 / +0.008 | 0.051 / 0.143 / **0.042** / 0.127 |
| `christian` | 0.932 / **0.934** / 0.906 / 0.911 | +0.003 / **−0.003** / +0.006 / +0.002 | 0.015 / 0.026 / 0.022 / **0.007** |

---

**Overall performance.** AUC = 0.9497 (lowest of the four). Predicted toxic rate 5.70% — more conservative than adversarial (5.96%) but less so than reweighting (4.80%). Recall 55% and precision 76%; stacking the two objectives introduces training pressure that modestly hurts the toxicity classifier's discrimination.

**AUC.** Combined achieves the best avg BPSN AUC (0.905), marginally edging reweighting (0.904) and substantially improving over adversarial (0.880) and baseline (0.876). This means non-toxic identity-mentioning comments are least likely to be ranked above toxic background comments under the combined model. Conversely, avg BNSP AUC (0.920) is the lowest of the four — the model slightly under-detects toxic content within identity subgroups, a consequence of the conservative reweighting pressure.

**FPR.** Avg FPR gap (0.007) sits between reweighting (0.003) and baseline (0.022): an improvement over adversarial (0.024) but not as strong as reweighting alone. The reweighting component is the dominant driver of FPR reduction here — the adversarial component adds noise rather than further reducing over-triggering.

**ECE.** Overall ECE (0.007) and avg subgroup ECE (0.083) are both worse than adversarial alone (0.003 / 0.058) but better than reweighting (0.012 / 0.089). The adversarial component improves calibration relative to pure reweighting, but the importance-weighted loss pulls probability mass downward, partially counteracting the calibration benefit. Notable exception: `christian` ECE drops to 0.007 (best of all four models), likely because the mild upscaling of christian toxic comments under reweighting aligns its predicted score distribution with its true toxic rate.

**Counterfactual gap.** Avg neutral CF gap (0.014) ties with reweighting as the best. However, the ranking of identities shifts: `christian` → `person` now produces the largest gap (0.034), overtaking `black` (0.029) and `transgender` (0.028). This is likely an interaction effect — reweighting slightly upscales christian toxic-comment losses, and the adversary's pressure on the encoder redirects identity-token associations rather than eliminating them.

**Takeaway.** Stacking reweighting with adversarial debiasing achieves the best avg BPSN AUC and ties for the best neutral CF gap, confirming that the two methods address complementary failure modes. FPR improves relative to adversarial alone; calibration improves relative to reweighting alone. The cost is the lowest overall AUC (0.9497) and BNSP AUC (0.920) of the four models. No single model dominates across all fairness dimensions; the combined approach offers the best non-toxic-identity discrimination (BPSN) and CF gap reduction, at the expense of overall classification accuracy.